# What's resolution for?

The resolution of a friction surface is usually defined by the resolution of the input landcover data, or by the compute limitations of the analysis. This notebook compares the results of using two different datasets:

- The Global Friction Surface from the Malaria Atlas Project (the default)
- A new 30m resolution dataset from the same group, produced by GOST's own Walker Bradley

In [1]:
import sys
import os
import rasterio

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box

sys.path.insert(0, r"C:\WBG\Work\Code\GOSTrocks\src")
import GOSTrocks.rasterMisc as rMisc
from GOSTrocks.misc import tPrint

sys.path.append(r"C:\WBG\Work\Code\GOSTnetsraster\src")
import GOSTnetsraster.market_access as ma
import GOSTnetsraster.conversion_tables as speed_tables

%load_ext autoreload
%autoreload 2

GDAL is not installed - OGR functionality not available


In [2]:
high_res_path = "C:\\WBG\\Work\\data\\FRICTION\\WALKER\\ForJRC\\ITA_GRUFS_mx.tif"
low_res_path = "https://datacatalogfiles.worldbank.org/ddh-published/0066950/DR0095710/2020_motorized_friction_surface.geotiff"

processing_folder = "C:/Temp/res_comparison"
local_low_res_file = os.path.join(processing_folder, "low_res.tif")
local_high_res_file = os.path.join(processing_folder, "high_res.tif")

if not os.path.exists(processing_folder):
    os.makedirs(processing_folder)

high_res_r = rasterio.open(high_res_path)
low_res_r = rasterio.open(low_res_path)

In [3]:
high_res_r.meta

{'driver': 'GTiff',
 'dtype': 'float32',
 'nodata': -3.4028234663852886e+38,
 'width': 61421,
 'height': 54376,
 'count': 1,
 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]'),
 'transform': Affine(0.0002499999999999999, 0.0, 5.475500000000011,
        0.0, -0.0002499999999999999, 48.14574999999999)}

In [5]:
low_res_r.meta

{'driver': 'GTiff',
 'dtype': 'float32',
 'nodata': -9999.0,
 'width': 43200,
 'height': 17400,
 'count': 1,
 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]'),
 'transform': Affine(0.008333333333333333, 0.0, -180.0,
        0.0, -0.008333333333333333, 85.0)}

In [11]:
def clip_compress_raster(in_path, out_path, bounds):
    with rasterio.open(in_path) as src:
        out_image, out_transform = rasterio.mask.mask(src, [box(*bounds)], crop=True)
        out_image = (out_image * 10000).astype(rasterio.uint16)
        out_image[out_image<=0] = np.iinfo(np.uint16).max        
        out_meta = src.meta.copy()
        out_meta.update({"driver": "GTiff",
                         "height": out_image.shape[1],
                         "width": out_image.shape[2],
                         "transform": out_transform,
                         "compress":'zstd',
                         "predictor":2,
                         "tiled":True,
                         "dtype":"uint16",
                         "nodata":np.iinfo(np.uint16).max
                         })
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

if not os.path.exists(local_low_res_file):
    tPrint("Downloading low res raster...")
    clip_compress_raster(low_res_path, local_low_res_file, bounds=high_res_r.bounds)

if not os.path.exists(local_high_res_file):
    tPrint("Downloading high res raster...")
    clip_compress_raster(high_res_path, local_high_res_file, bounds=high_res_r.bounds)

15:29:43	Downloading low res raster...
15:29:43	Downloading high res raster...


C:\Users\WB411133\AppData\Local\Temp\ipykernel_20036\1393507025.py:4: RuntimeWarning: overflow encountered in multiply
  out_image = (out_image * 10000).astype(rasterio.uint16)
C:\Users\WB411133\AppData\Local\Temp\ipykernel_20036\1393507025.py:4: RuntimeWarning: invalid value encountered in cast
  out_image = (out_image * 10000).astype(rasterio.uint16)


In [13]:
def describe_raster(path):    
    inR = rasterio.open(path)
    inD = inR.read(1)
    # inD[inD<=0] = np.nan
    print(inD.shape)
    print(f'{np.nanmin(inD)} to {np.nanmax(inD)}')
    return(inR)

high_res_r = describe_raster(local_high_res_file)
low_res_r = describe_raster(local_low_res_file)

(54376, 61421)
4 to 65535
(1632, 1843)
4 to 17936
